# Sprint 10: Clasificación y Churn Bancario

# Proyecto: Recomendación de planes para Megaline
## Smart o Ultra
### En este proyecto, nuestro objetivo es desarrollar un modelo de clasificación para la compañía móvil Megaline que analice el comportamiento de los clientes y recomiende uno de los nuevos planes: Smart (0) o Ultra (1). El umbral de exactitud requerido es de al menos 0.75.

In [1]:
# Carga de librerías necesarias
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyClassifier

## 1. Abre y examina el archivo de datos
### Cargaremos el dataset proporcionado y realizaremos una inspección inicial para conocer su estructura, tipos de datos y verificar si existen valores nulos o duplicados.

In [2]:
import os
import pandas as pd

# Ruta robusta para compatibilidad tanto en la plataforma de TripleTen como en entorno local
path = '/datasets/users_behavior.csv'
if not os.path.exists(path):
    path = 'users_behavior.csv'

# Leer el archivo de datos
df = pd.read_csv(path)

# Mostrar las primeras 5 filas
display(df.head())

# Información general del DataFrame
df.info()

# Verificar valores ausentes o duplicados
print("\nValores nulos por columna:")
print(df.isnull().sum())
print("\nCantidad de filas duplicadas:", df.duplicated().sum())

,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB

Valores nulos por columna:
calls       0
minutes     0
messages    0
mb_used     0
is_ultra    0
dtype: int64

Cantidad de filas duplicadas: 0


## 2. Segmenta los datos fuente
Dividiremos el conjunto de datos original en tres partes para evitar el sobreajuste y evaluar de manera imparcial:

Conjunto de entrenamiento (Train): 60% de los datos (para entrenar los modelos).

Conjunto de validación (Validation): 20% de los datos (para sintonizar hiperparámetros).

Conjunto de prueba (Test): 20% de los datos (para la evaluación final independiente).

Utilizaremos el parámetro stratify para mantener la proporción de las clases del objetivo (is_ultra).

In [3]:
# Separar características (features) y objetivo (target)
X = df.drop('is_ultra', axis=1)
y = df['is_ultra']

# Primer división: Separar el conjunto de prueba (20%) del conjunto de entrenamiento + validación (80%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Segunda división: Separar el conjunto de entrenamiento (75% de los 80% restante = 60% total) y validación (20% total)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

# Verificar los tamaños de los conjuntos resultantes
print(f"Tamaño del conjunto de entrenamiento: {X_train.shape[0]} filas ({X_train.shape[0] / len(df):.0%})")
print(f"Tamaño del conjunto de validación: {X_val.shape[0]} filas ({X_val.shape[0] / len(df):.0%})")
print(f"Tamaño del conjunto de prueba: {X_test.shape[0]} filas ({X_test.shape[0] / len(df):.0%})")

Tamaño del conjunto de entrenamiento: 1928 filas (60%)
Tamaño del conjunto de validación: 643 filas (20%)
Tamaño del conjunto de prueba: 643 filas (20%)


## 3. Investiga la calidad de diferentes modelos cambiando los hiperparámetros
Evaluaremos tres algoritmos de clasificación diferentes:

Árbol de Decisión (DecisionTreeClassifier): variando la profundidad máxima (max_depth).

Bosque Aleatorio (RandomForestClassifier): variando el número de estimadores (n_estimators) y la profundidad máxima (max_depth).

Regresión Logística (LogisticRegression): como modelo lineal base.

In [4]:
# 1. Árbol de Decisión
best_tree_acc = 0
best_tree_depth = 0

for depth in range(1, 16):
    model_tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model_tree.fit(X_train, y_train)
    val_preds = model_tree.predict(X_val)
    acc = accuracy_score(y_val, val_preds)
    
    if acc > best_tree_acc:
        best_tree_acc = acc
        best_tree_depth = depth

print(f"Mejor Árbol de Decisión - Exactitud: {best_tree_acc:.4f} (max_depth={best_tree_depth})")

# 2. Bosque Aleatorio
best_rf_acc = 0
best_rf_params = {}

for est in range(10, 61, 10):
    for depth in range(1, 11):
        model_rf = RandomForestClassifier(n_estimators=est, max_depth=depth, random_state=42)
        model_rf.fit(X_train, y_train)
        val_preds = model_rf.predict(X_val)
        acc = accuracy_score(y_val, val_preds)
        
        if acc > best_rf_acc:
            best_rf_acc = acc
            best_rf_params = {'n_estimators': est, 'max_depth': depth}

print(f"Mejor Bosque Aleatorio - Exactitud: {best_rf_acc:.4f} con hiperparámetros: {best_rf_params}")

# 3. Regresión Logística
model_lr = LogisticRegression(random_state=42)
model_lr.fit(X_train, y_train)
lr_acc = accuracy_score(y_val, model_lr.predict(X_val))
print(f"Regresión Logística - Exactitud: {lr_acc:.4f}")

Mejor Árbol de Decisión - Exactitud: 0.7900 (max_depth=5)
Mejor Bosque Aleatorio - Exactitud: 0.7963 con hiperparámetros: {'n_estimators': 30, 'max_depth': 7}
Regresión Logística - Exactitud: 0.7449


## Hallazgos del estudio:
Árbol de Decisión: Alcanza un buen rendimiento con profundidades moderadas (en torno a max_depth = 5). Si el árbol crece demasiado, comienza a sufrir de sobreajuste (overfitting).

Bosque Aleatorio (RandomForestClassifier): Demuestra el mejor desempeño general al combinar múltiples árboles, superando consistentemente el umbral de 0.75 en el conjunto de validación.

Regresión Logística: Muestra menor precisión (~0.74), lo cual indica que las fronteras de decisión entre las características de comportamiento de los usuarios no son completamente lineales.

## 4. Comprueba la calidad del modelo usando el conjunto de prueba
Entrenaremos el mejor modelo encontrado (Bosque Aleatorio con los hiperparámetros óptimos) uniendo los conjuntos de entrenamiento y validación para aprovechar todo el histórico disponible, y evaluaremos su desempeño final en el conjunto de prueba (X_test, y_test).

In [5]:
# Combinar entrenamiento y validación para el entrenamiento final del mejor modelo
X_train_val_final = pd.concat([X_train, X_val])
y_train_val_final = pd.concat([y_train, y_val])

# Instanciar el modelo con los mejores hiperparámetros encontrados
final_model = RandomForestClassifier(
    n_estimators=best_rf_params['n_estimators'],
    max_depth=best_rf_params['max_depth'],
    random_state=42
)

# Entrenar el modelo
final_model.fit(X_train_val_final, y_train_val_final)

# Evaluar en el conjunto de prueba
test_predictions = final_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)

print(f"Exactitud final del modelo en el conjunto de prueba: {test_accuracy:.4f}")

# Verificar si cumple con el umbral requerido
if test_accuracy >= 0.75:
    print("¡El modelo cumple y supera el umbral de exactitud de 0.75!")
else:
    print("El modelo no alcanza el umbral de exactitud requerido.")

Exactitud final del modelo en el conjunto de prueba: 0.8180
¡El modelo cumple y supera el umbral de exactitud de 0.75!


## 5. Tarea adicional: Prueba de cordura (Sanity Check)
Para comprobar que nuestro modelo realmente está aprendiendo patrones útiles y no está simplemente prediciendo la clase mayoritaria, utilizaremos un DummyClassifier como línea base de control.

In [6]:
# Crear y entrenar un clasificador tonto (Dummy Classifier) con la estrategia de la clase más frecuente
dummy_model = DummyClassifier(strategy='most_frequent', random_state=42)
dummy_model.fit(X_train_val_final, y_train_val_final)

# Evaluar el modelo dummy en el conjunto de prueba
dummy_predictions = dummy_model.predict(X_test)
dummy_accuracy = accuracy_score(y_test, dummy_predictions)

print(f"Exactitud del Dummy Classifier (Prueba de cordura): {dummy_accuracy:.4f}")
print(f"Exactitud de nuestro modelo de Bosque Aleatorio: {test_accuracy:.4f}")

if test_accuracy > dummy_accuracy:
    print("La prueba de cordura fue superada exitosamente: nuestro modelo rinde significativamente mejor que una estrategia trivial.")
else:
    print("Advertencia: El modelo no supera a la línea base.")

Exactitud del Dummy Classifier (Prueba de cordura): 0.6936
Exactitud de nuestro modelo de Bosque Aleatorio: 0.8180
La prueba de cordura fue superada exitosamente: nuestro modelo rinde significativamente mejor que una estrategia trivial.


## Paso 6: Ejemplo práctico de recomendación por usuario

Para ilustrar cómo opera el modelo en la práctica, generaremos una tabla comparativa entre el plan real que contrataron los primeros 10 usuarios del conjunto de prueba y el plan que nuestro modelo de **Bosque Aleatorio** les habría recomendado basándose en su comportamiento mensual.

In [7]:
# Crear un DataFrame para comparar las predicciones del modelo con el plan real
comparacion = pd.DataFrame({
    'Plan Real': y_test.values[:10],
    'Plan Recomendado (Modelo)': test_predictions[:10]
})

# Mapear los valores numéricos a los nombres de los planes para mayor claridad visual
comparacion['Plan Real'] = comparacion['Plan Real'].map({0: 'Smart', 1: 'Ultra'})
comparacion['Plan Recomendado (Modelo)'] = comparacion['Plan Recomendado (Modelo)'].map({0: 'Smart', 1: 'Ultra'})

print("Comparación de los primeros 10 clientes del conjunto de prueba:")
display(comparacion)

Comparación de los primeros 10 clientes del conjunto de prueba:


,Plan Real,Plan Recomendado (Modelo)
0,Smart,Smart
1,Smart,Ultra
2,Ultra,Ultra
3,Ultra,Ultra
4,Ultra,Smart
5,Ultra,Ultra
6,Ultra,Smart
7,Smart,Smart
8,Ultra,Smart
9,Ultra,Smart


## ¿Cómo leíste los datos después de descargarlos?
**Respuesta:** Empleamos `pandas` mediante la función `pd.read_csv()`. Implementamos una ruta robusta con `os.path.exists()` para asegurar la compatibilidad tanto en la plataforma de TripleTen como en entorno local. Finalmente, realizamos una inspección visual con `head()`, revisamos su estructura con `info()` y verificamos que no existieran valores nulos ni filas duplicadas.

## ¿Segmentaste correctamente los datos en conjuntos de entrenamiento, validación y prueba?
**Respuesta:** **Sí.** El dataset original fue segmentado rigurosamente en tres subconjuntos utilizando la función `train_test_split` de `scikit-learn` en dos etapas. Además, aplicamos el parámetro `stratify=y` en ambas divisiones para garantizar que la proporción de las clases (planes Smart y Ultra) se mantuviera equilibrada en cada conjunto.

## ¿Cómo escogiste el tamaño de los conjuntos?
**Respuesta:** Los tamaños se distribuyeron siguiendo la proporción estándar recomendada:
* **Entrenamiento (60%):** Proporciona la mayor cantidad de información para que los modelos aprendan los patrones de comportamiento.
* **Validación (20%):** Permite sintonizar y comparar los hiperparámetros sin sesgar la evaluación final.
* **Prueba (20%):** Se mantuvo completamente aislado durante la fase de entrenamiento y ajuste para realizar la prueba de rendimiento final de forma totalmente imparcial.

## ¿Evaluaste correctamente la calidad del modelo?
**Respuesta:** **Sí.** La calidad se evaluó utilizando la métrica de exactitud (*accuracy*) mediante la función `accuracy_score` de `scikit-learn`. Comparamos las predicciones generadas frente a las etiquetas reales de los conjuntos de validación y de prueba.

## ¿Qué modelos e hiperparámentros usaste?
**Respuesta:** Evaluamos tres algoritmos diferentes ajustando sus hiperparámetros clave:
* **Árbol de Decisión (`DecisionTreeClassifier`):** Evaluando profundidades máximas (`max_depth`) en un rango de 1 a 15.
* **Bosque Aleatorio (`RandomForestClassifier`):** Evaluando combinaciones de número de estimadores (`n_estimators` de 10 a 60) y profundidad máxima (`max_depth` de 1 a 10).
* **Regresión Logística (`LogisticRegression`):** Utilizada como modelo lineal base con sus parámetros predeterminados.

## ¿Cuáles fueron tus hallazgos?
**Respuesta:** 
* El **Árbol de Decisión** rinde bien con profundidades moderadas (encontrando su óptimo en `max_depth = 5`), pero su exactitud decae si el árbol crece demasiado debido al sobreajuste (*overfitting*).
* El **Bosque Aleatorio** demostró ser el mejor modelo de todos, ya que al combinar múltiples árboles reduce la varianza y supera de manera consistente el umbral de calidad requerido.
* La **Regresión Logística** obtuvo la menor precisión (~0.74), lo que indica que las fronteras entre el comportamiento de los clientes y la elección del plan no son lineales.

## ¿Probaste los modelos correctamente?
**Respuesta:** **Sí.** Primero sintonizamos los hiperparámetros usando el conjunto de validación. Luego, combinamos los datos de entrenamiento y validación (`X_train_val_final`) para maximizar el histórico disponible, entrenamos el modelo final y realizamos la prueba estricta sobre el conjunto de prueba (`X_test`). Adicionalmente, realizamos una prueba de cordura (*Sanity Check*) usando un `DummyClassifier`.

## ¿Cuál es tu puntuación de exactitud?
**Respuesta:** La exactitud final obtenida por nuestro modelo de Bosque Aleatorio en el conjunto de prueba fue de **0.8180** (81.8%), superando holgadamente el umbral mínimo exigido de **0.75**.

## ¿Te ceñiste a la estructura del proyecto y mantuviste limpio el código?
**Respuesta:** **Sí.** El cuaderno (`.ipynb`) mantiene una estructura modular y secuencial paso a paso (Apertura, Segmentación, Investigación de modelos, Evaluación final y Prueba de cordura), acompañada de celdas de Markdown bien redactadas y código limpio.
Incluyendo un ejemplo práctico al final.